# 07   Graph Exploration & GraphRAG Trace

Spec: *Notebook 3   Graph exploration*
(`Build_a_Local_GraphRAG_System_with_neo4j-gr.md`): useful Cypher over the
**actual schema**   highly connected entities, relationship counts, entity
types, neighborhoods, source provenance   plus a live **question -> documents**
trace through `GraphRAG.query()` that yields a paste-ready Neo4j Browser query.

**Ground rules**

* Read-only against live Neo4j (bolt `:7687`, database `neo4j`),
  2,049 nodes / 4,139 edges. Section F asserts nothing changed.
* No new indexes or constraints are created (the `label(n)` scans are fine
  at this size; `lineage_id IS UNIQUE` indexes already exist   see A.4).
* Visualisation = the built-in **Neo4j Browser** at `http://localhost:7474`.
  Every section prints Cypher you can copy straight in.
* Section **E** is the only section that calls the LLM (Ollama `qwen3.8:27b`);
  E.4's refusal is deterministic (no LLM call   `rag.py:246`).

**Schema in one line** (verified in A)

`Document / ExternalDocument / Article / ExternalArticle / Preamble / Term / Entity`
  `DEFINED_IN   CROSS_REFERENCES   AMENDS   APPLIES_TO`
  every node carries `lineage_id` (`doc_slug:kind:slug`) and `doc_id`.

## 0   Setup

Connect read-only, confirm the stack, define the helpers, and set the two
knobs for the parameterised sections: `DOC` (section C) and `QUESTION` (E).

In [1]:
import sys, os
from pathlib import Path

def _repo_root():
    cur = Path.cwd().resolve()
    for cand in [cur, *cur.parents]:
        if (cand / "notebooks").is_dir() and (cand / "data").is_dir():
            return cand
    return cur

ROOT = _repo_root()
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

import neo4j_config as cfg
from graphrag_n4j import KNOWN_EDGE_KINDS
from graphrag_n4j.schema import NODE_LABELS
import common
from graphrag_n4j import GraphRAG
import pandas as pd

driver = cfg.make_driver()
DB = cfg.neo4j_settings().database          # 'neo4j'
print("connected:", bool(driver.verify_connectivity()))
with driver.session(database=DB) as s:
    n = s.run("MATCH (n) RETURN count(n) AS c").single()["c"]
    e = s.run("MATCH ()-[r]->() RETURN count(r) AS c").single()["c"]
    print("graph    :", n, "nodes /", e, "edges")
print("labels    :", NODE_LABELS)
print("edge kinds:", sorted(KNOWN_EDGE_KINDS))

connected: False
graph    : 2049 nodes / 4139 edges
labels    : ('Document', 'ExternalDocument', 'Article', 'ExternalArticle', 'Preamble', 'Term', 'Entity')
edge kinds: ['AMENDS', 'APPLIES_TO', 'CROSS_REFERENCES', 'DEFINED_IN']


In [2]:
def run(query: str, **params) -> list:
    '''Execute read-only Cypher, return a list of row-dicts.'''
    with driver.session(database=DB) as s:
        return [dict(r) for r in s.run(query, **params)]


def show(rows: list, limit: int = 15) -> None:
    for r in rows[:limit]:
        # keep embeddings out of the printed rows
        print({k: v for k, v in r.items() if "emb" not in str(k) and not str(k).endswith(".emb")})
    if len(rows) > limit:
        print(f"... and {len(rows) - limit} more row(s)")

In [3]:
# ---- knobs for the parameterised sections ----
# C: document slug for the neighborhood deep-dive (None -> C.1 prints all docs)
DOC = "elec_dir_2019_944"

# E: the question run through GraphRAG.query()
QUESTION = ("Who must provide ancillary services in the electricity market, "
            "and which article defines the term?")

## A   Schema & inventory

(entity types, relationship counts   the spec's required overview.)

In [4]:
show(run(
    'MATCH (n) '
    'RETURN labels(n)[0] AS label, count(n) AS nodes '
    'ORDER BY nodes DESC'
))

{'label': 'Article', 'nodes': 1021}
{'label': 'Term', 'nodes': 839}
{'label': 'ExternalArticle', 'nodes': 119}
{'label': 'Document', 'nodes': 25}
{'label': 'ExternalDocument', 'nodes': 25}
{'label': 'Preamble', 'nodes': 10}
{'label': 'Entity', 'nodes': 10}


In [5]:
show(run(
    'MATCH (a)-[r]->(b) '
    'RETURN type(r) AS kind, count(r) AS edges '
    'ORDER BY edges DESC'
))

{'kind': 'CROSS_REFERENCES', 'edges': 3186}
{'kind': 'DEFINED_IN', 'edges': 839}
{'kind': 'APPLIES_TO', 'edges': 60}
{'kind': 'AMENDS', 'edges': 54}


In [6]:
# one sample node per label (properties only, embeddings filtered by show())
q = ('UNWIND [' + ','.join('"%s"' % l for l in
     ["Document", "ExternalDocument", "Article", "ExternalArticle", "Preamble", "Term", "Entity"])
     + '] AS l '
     'OPTIONAL MATCH (n) WHERE l IN labels(n) '
     'WITH l, head(collect(n)) AS s '
     'RETURN l, [k IN keys(s) WHERE k <> "emb"] AS properties')
rows = run(q)
for r in rows:
    print(f"{r['l']:16s} {r['properties']}")

Document         ['kind', 'title', 'doc_id', 'search_text', 'lineage_id']
ExternalDocument ['kind', 'search_text', 'title', 'lineage_id']
Article          ['kind', 'doc_id', 'number', 'search_text', 'title', 'lineage_id']
ExternalArticle  ['kind', 'title', 'lineage_id', 'search_text', 'number']
Preamble         ['doc_id', 'title', 'kind', 'search_text', 'lineage_id']
Term             ['term', 'doc_id', 'kind', 'lineage_id', 'search_text']
Entity           ['label', 'kind', 'search_text', 'lineage_id']


In [7]:
# constraints + indexes that make lineage_id / vector / fulltext lookups cheap
c = run('SHOW CONSTRAINTS YIELD name, labelsOrTypes, type')
print(f"{len(c)} constraints; e.g.:", c[:2])
v = run('SHOW VECTOR INDEXES YIELD name, labelsOrTypes, properties, state, populationPercent')
print(f"{len(v)} vector indexes; e.g.:", v[:2])
f = run('SHOW FULLTEXT INDEXES YIELD name, labelsOrTypes, properties, state')
print(f"{len(f)} fulltext indexes; e.g.:", f[:2])

7 constraints; e.g.: [{'name': 'lineage_unique_Article', 'labelsOrTypes': ['Article'], 'type': 'UNIQUENESS'}, {'name': 'lineage_unique_Document', 'labelsOrTypes': ['Document'], 'type': 'UNIQUENESS'}]
2 vector indexes; e.g.: [{'name': 'v_article_emb', 'labelsOrTypes': ['Article'], 'properties': ['emb'], 'state': 'ONLINE', 'populationPercent': 100.0}, {'name': 'v_term_emb', 'labelsOrTypes': ['Term'], 'properties': ['emb'], 'state': 'ONLINE', 'populationPercent': 100.0}]
1 fulltext indexes; e.g.: [{'name': 'ft_term', 'labelsOrTypes': ['Term'], 'properties': ['search_text'], 'state': 'ONLINE'}]


## B   Connectivity

(highly connected entities; plus a data-quality pass.)

In [8]:
show(run(
    'MATCH (n) '
    'OPTIONAL MATCH (n)-[r]->() '
    'RETURN labels(n)[0] AS label, n.lineage_id AS lineage_id, count(r) AS links '
    'ORDER BY links DESC '
    'LIMIT 10'
), limit=10)

{'label': 'Article', 'lineage_id': 'entso_sogl_2017_1485:article:192', 'links': 40}
{'label': 'Article', 'lineage_id': 'eidas_2_2024_1183:article:16', 'links': 36}
{'label': 'Article', 'lineage_id': 'entso_cacm_2015_1222:article:9', 'links': 32}
{'label': 'Article', 'lineage_id': 'emd_reform_dir_2024_1711:article:2', 'links': 29}
{'label': 'Article', 'lineage_id': 'entso_sogl_2017_1485:article:6', 'links': 29}
{'label': 'Article', 'lineage_id': 'entso_ebgl_2017_2195:article:5', 'links': 27}
{'label': 'Article', 'lineage_id': 'entso_sogl_2017_1485:article:118', 'links': 25}
{'label': 'Article', 'lineage_id': 'elec_reg_2019_943:article:64', 'links': 25}
{'label': 'Article', 'lineage_id': 'emd_reform_reg_2024_1747:article:19h', 'links': 24}
{'label': 'Article', 'lineage_id': 'elec_dir_2019_944:article:71', 'links': 24}


In [9]:
# which node *type* holds the hubs? (per-label leader)
show(run(
    'MATCH (n) '
    'OPTIONAL MATCH (n)-[r]->() '
    'WITH labels(n)[0] AS label, n AS node, count(r) AS links '
    'WITH label, max(links) AS best, collect({node: node, links: links}) AS all '
    'RETURN label, [a IN all WHERE a.links = best][0].node.lineage_id AS hub, best AS links '
    'ORDER BY best DESC'
))

{'label': 'Article', 'hub': 'entso_sogl_2017_1485:article:192', 'links': 40}
{'label': 'Document', 'hub': 'acer_remit_guidance:document', 'links': 10}
{'label': 'Preamble', 'hub': 'eu_ai_act_2024_1689:preamble', 'links': 7}
{'label': 'Term', 'hub': 'entso_ncrfg_2016_631:term:minimum_stable_operating_level', 'links': 1}
{'label': 'ExternalArticle', 'hub': 'ext:article:1', 'links': 0}
{'label': 'ExternalDocument', 'hub': 'ext:instrument:1025-2012', 'links': 0}
{'label': 'Entity', 'hub': 'corpus:entity:balance_responsible_parties', 'links': 0}


In [10]:
# data quality: isolated nodes and edges missing lineage ids (expect clean)
iso = run('MATCH (n) WHERE NOT (n)--() RETURN labels(n)[0] AS label, count(n) AS c')
dang = run(
    'MATCH (a)-[r]->(b) '
    'WHERE a.lineage_id IS NULL OR b.lineage_id IS NULL '
    'RETURN count(r) AS c'
)
print("isolated nodes:", iso or "none")
print("dangling edges:", dang)

isolated nodes: [{'label': 'Document', 'c': 20}]
dangling edges: [{'c': 0}]


## C   Neighborhood deep-dive (parameterised by `DOC`)

C.0 lists every document so you can swap `DOC` in the setup cell and re-run
this section. Each block is also a paste-ready browser query.

> `Document` nodes are provenance anchors   most edges live between that
> document's `Article` / `Term` / `Entity` nodes and their external
> counterparts, which is why the Document node itself shows low degree.
> The blocks below are the interesting part.

In [11]:
q = ('MATCH (d:Document) '
     'RETURN d.doc_id AS doc_id, d.title AS title '
     'ORDER BY d.doc_id')
rows = run(q)
print(f"{len(rows)} documents   C is running with DOC = {DOC!r}")
for r in rows:
    mark = "   <- DOC" if r["doc_id"] == DOC else ""
    print(f"  {r['doc_id']:36s}{mark}")

25 documents   C is running with DOC = 'elec_dir_2019_944'
  acer_remit_guidance                 
  data_act_2023_2854                  
  dlt_pilot_2022_858                  
  dora_2022_2554                      
  eidas_2014_910                      
  eidas_2_2024_1183                   
  elec_dir_2019_944                      <- DOC
  elec_reg_2019_943                   
  emd_reform_dir_2024_1711            
  emd_reform_reg_2024_1747            
  entso-e_compliance_monitoring       
  entso-e_simulation_models           
  entso_cacm_2015_1222                
  entso_ebgl_2017_2195                
  entso_ncrfg_2016_631                
  entso_sogl_2017_1485                
  eprivacy_dir_2002_58                
  eu_ai_act_2024_1689                 
  gdpr_2016_679                       
  know_your_contract_guidance         
  metering_data_2023_1162             
  mica_2023_1114                      
  nis2_dir_2022_2555                  
  remit_1227_2011                  

In [12]:
# C.1 the document's articles, with how many terms each defines
show(run(
    'MATCH (a:Article {doc_id: $doc}) '
    'OPTIONAL MATCH (t:Term)-[:DEFINED_IN]->(a) '
    'RETURN a.number AS article, a.title AS title, count(DISTINCT t) AS terms '
    'ORDER BY a.number '
    'LIMIT 15',
    doc=DOC
))

{'article': '10', 'title': 'Article 10', 'terms': 0}
{'article': '11', 'title': 'Entitlement to a dynamic electricity price contract', 'terms': 0}
{'article': '12', 'title': 'Right to switch and rules on switching-related fees', 'terms': 0}
{'article': '13', 'title': 'Aggregation contract', 'terms': 0}
{'article': '14', 'title': 'Comparison tools', 'terms': 0}
{'article': '15', 'title': 'Active customers', 'terms': 0}
{'article': '16', 'title': 'Article 16', 'terms': 0}
{'article': '17', 'title': 'Demand response through aggregation', 'terms': 0}
{'article': '19', 'title': 'Smart metering systems', 'terms': 1}
{'article': '2', 'title': 'Article 2', 'terms': 60}
{'article': '20', 'title': 'Functionalities of smart metering systems', 'terms': 0}
{'article': '21', 'title': 'Entitlement to a smart meter', 'terms': 0}
{'article': '23', 'title': 'Data management', 'terms': 0}
{'article': '24', 'title': 'Interoperability requirements and procedures for access to data', 'terms': 0}
{'article':

In [13]:
# C.2 DEFINED_IN: the document's defined terms
show(run(
    'MATCH (t:Term {doc_id: $doc})-[:DEFINED_IN]->(a:Article) '
    'RETURN t.term AS term, a.number AS article, a.title AS title '
    'ORDER BY a.number, t.term '
    'LIMIT 15',
    doc=DOC
))

{'term': 'start of works', 'article': '19', 'title': 'Smart metering systems'}
{'term': 'active customer', 'article': '2', 'title': 'Article 2'}
{'term': 'aggregation', 'article': '2', 'title': 'Article 2'}
{'term': 'ancillary service', 'article': '2', 'title': 'Article 2'}
{'term': 'balance responsible party', 'article': '2', 'title': 'Article 2'}
{'term': 'balancing', 'article': '2', 'title': 'Article 2'}
{'term': 'balancing energy', 'article': '2', 'title': 'Article 2'}
{'term': 'best available techniques', 'article': '2', 'title': 'Article 2'}
{'term': 'billing information', 'article': '2', 'title': 'Article 2'}
{'term': 'citizen energy community', 'article': '2', 'title': 'Article 2'}
{'term': 'congestion', 'article': '2', 'title': 'Article 2'}
{'term': 'contract termination fee', 'article': '2', 'title': 'Article 2'}
{'term': 'control', 'article': '2', 'title': 'Article 2'}
{'term': 'conventional meter', 'article': '2', 'title': 'Article 2'}
{'term': 'customer', 'article': '2', '

In [14]:
# C.3 CROSS_REFERENCES: what this document cites, aggregated by target
show(run(
    'MATCH (a)-[:CROSS_REFERENCES]->(o) '
    'WHERE a.doc_id = $doc '
    'RETURN labels(o)[0] AS kind, o.doc_id AS other_doc, '
    '       o.number AS article, o.lineage_id AS target, count(*) AS mentions '
    'ORDER BY mentions DESC '
    'LIMIT 10',
    doc=DOC
))
print()
show(run(
    'MATCH (a)-[:CROSS_REFERENCES]->(o) '
    'WHERE o.doc_id = $doc '
    'RETURN labels(a)[0] AS kind, a.doc_id AS citing_doc, '
    '       a.number AS article, count(*) AS mentions '
    'ORDER BY mentions DESC '
    'LIMIT 10',
    doc=DOC
))

{'kind': 'Article', 'other_doc': 'elec_dir_2019_944', 'article': '59', 'target': 'elec_dir_2019_944:article:59', 'mentions': 10}
{'kind': 'Article', 'other_doc': 'elec_dir_2019_944', 'article': '9', 'target': 'elec_dir_2019_944:article:9', 'mentions': 7}
{'kind': 'Article', 'other_doc': 'elec_dir_2019_944', 'article': '6', 'target': 'elec_dir_2019_944:article:6', 'mentions': 7}
{'kind': 'Article', 'other_doc': 'elec_dir_2019_944', 'article': '43', 'target': 'elec_dir_2019_944:article:43', 'mentions': 6}
{'kind': 'Article', 'other_doc': 'elec_dir_2019_944', 'article': '7', 'target': 'elec_dir_2019_944:article:7', 'mentions': 5}
{'kind': 'Article', 'other_doc': 'elec_dir_2019_944', 'article': '5', 'target': 'elec_dir_2019_944:article:5', 'mentions': 5}
{'kind': 'Article', 'other_doc': 'elec_dir_2019_944', 'article': '35', 'target': 'elec_dir_2019_944:article:35', 'mentions': 4}
{'kind': 'Article', 'other_doc': 'elec_reg_2019_943', 'article': '18', 'target': 'elec_reg_2019_943:article:18'

In [15]:
# C.4 AMENDS: instruments this document amends, and that amend it
show(run(
    'MATCH (a)-[r:AMENDS]->(b) '
    'WHERE a.doc_id = $doc OR b.doc_id = $doc '
    'RETURN a.lineage_id AS from, b.lineage_id AS to '
    'LIMIT 15',
    doc=DOC
))

{'from': 'acer_remit_guidance:document', 'to': 'elec_dir_2019_944:document'}
{'from': 'elec_dir_2019_944:article:11', 'to': 'ext:instrument:1093-2010'}
{'from': 'emd_reform_dir_2024_1711:article:2', 'to': 'elec_dir_2019_944:document'}


In [16]:
# C.5 APPLIES_TO: which actors/territories scope this document's articles
show(run(
    'MATCH (s)-[r:APPLIES_TO]->(e) '
    'WHERE s.doc_id = $doc OR e.doc_id = $doc '
    'RETURN s.lineage_id AS source, e.lineage_id AS entity '
    'LIMIT 15',
    doc=DOC
))

{'source': 'elec_dir_2019_944:article:40', 'entity': 'corpus:entity:transmission_system_operators'}
{'source': 'elec_dir_2019_944:article:40', 'entity': 'corpus:entity:member_states'}
{'source': 'elec_dir_2019_944:article:53', 'entity': 'corpus:entity:member_states'}
{'source': 'elec_dir_2019_944:article:74', 'entity': 'corpus:entity:member_states'}


In [17]:
# C.6 1-hop neighborhood of the document's BUSIEST article + 2-hop query for the browser
lid = run(
    'MATCH (a:Article {doc_id: $doc}) '
    'OPTIONAL MATCH (a)-[r]->() '
    'WITH a, count(r) AS links '
    'RETURN a.lineage_id AS lid, links '
    'ORDER BY links DESC '
    'LIMIT 1',
    doc=DOC
)[0]["lid"]
print("busiest article in the document:", lid)
show(run(
    'MATCH (a)-[r]->(b) WHERE a.lineage_id = $lid '
    'RETURN type(r) AS kind, labels(b)[0] AS kind_of_b, b.lineage_id AS target '
    'LIMIT 20',
    lid=lid
))
rows2 = run(
    'MATCH (a)-[r]->(b)-[r2]->(c) WHERE a.lineage_id = $lid '
    'RETURN count(c) AS two_hop_nodes',
    lid=lid
)
print()
print("== PASTE INTO NEO4J BROWSER (http://localhost:7474) ==")
print("MATCH (a)-[r1]->(b)-[r2]->(c) WHERE a.lineage_id = '" + lid + "'")
print("RETURN a, r1, b, r2, c")
print("LIMIT 1000")
print("(2-hop neighbourhood size:", rows2[0]["two_hop_nodes"], "end-nodes)")

busiest article in the document: elec_dir_2019_944:article:71


{'kind': 'CROSS_REFERENCES', 'kind_of_b': 'Article', 'target': 'elec_dir_2019_944:article:26'}
{'kind': 'CROSS_REFERENCES', 'kind_of_b': 'Article', 'target': 'elec_dir_2019_944:article:51'}
{'kind': 'CROSS_REFERENCES', 'kind_of_b': 'Article', 'target': 'elec_dir_2019_944:article:59'}
{'kind': 'CROSS_REFERENCES', 'kind_of_b': 'Article', 'target': 'elec_dir_2019_944:article:2'}
{'kind': 'CROSS_REFERENCES', 'kind_of_b': 'Article', 'target': 'elec_dir_2019_944:article:6'}
{'kind': 'CROSS_REFERENCES', 'kind_of_b': 'Article', 'target': 'elec_dir_2019_944:article:54'}
{'kind': 'CROSS_REFERENCES', 'kind_of_b': 'Article', 'target': 'elec_dir_2019_944:article:10'}
{'kind': 'CROSS_REFERENCES', 'kind_of_b': 'Article', 'target': 'elec_dir_2019_944:article:57'}
{'kind': 'CROSS_REFERENCES', 'kind_of_b': 'Article', 'target': 'elec_dir_2019_944:article:34'}
{'kind': 'CROSS_REFERENCES', 'kind_of_b': 'Article', 'target': 'elec_dir_2019_944:article:7'}
{'kind': 'CROSS_REFERENCES', 'kind_of_b': 'Article', 

## D   Provenance

`lineage_id` anatomy + a resolver back to the files on disk
(spec: *source provenance*).

In [18]:
lid = "elec_dir_2019_944:term:ancillary_service"
part = lid.split(":")
print("lineage_id :", lid)
print("  doc_id   :", part[0])
print("  kind     :", part[1])
print("  local id :", part[-1])
print()
show(run(
    'MATCH (t:Term {lineage_id: $lid}) '
    'OPTIONAL MATCH (t)-[:DEFINED_IN]->(a:Article) '
    'RETURN t.term AS term, t.doc_id AS doc_id, '
    '       a.number AS defined_in_article, a.title AS article_title',
    lid=lid
))

lineage_id : elec_dir_2019_944:term:ancillary_service
  doc_id   : elec_dir_2019_944
  kind     : term
  local id : ancillary_service



{'term': 'ancillary service', 'doc_id': 'elec_dir_2019_944', 'defined_in_article': '2', 'article_title': 'Article 2'}


In [19]:
md_files = sorted(common.PROCESSED_BASE.rglob(DOC + ".md"))
pdfs = sorted(common.RAW_BASE.rglob("*" + DOC + "*.pdf"))
print("processed md :", [str(p.relative_to(common.ROOT)) for p in md_files] or "not on disk")
print("raw pdf      :", [str(p.relative_to(common.ROOT)) for p in pdfs[:3]] or "none (guidance docs are raw-text)")

processed md : ['data/processed/eu/directives/elec_dir_2019_944.md']
raw pdf      : ['data/raw/eu/en/directives/elec_dir_2019_944.pdf']


In [20]:
# D.3 full inventory of the document's nodes (browser-ready, LIMIT 20)
show(run(
    'MATCH (n) WHERE n.doc_id = $doc '
    'RETURN labels(n)[0] AS kind, n.lineage_id AS lineage_id '
    'ORDER BY kind, lineage_id '
    'LIMIT 20',
    doc=DOC
))

{'kind': 'Article', 'lineage_id': 'elec_dir_2019_944:article:10'}
{'kind': 'Article', 'lineage_id': 'elec_dir_2019_944:article:11'}
{'kind': 'Article', 'lineage_id': 'elec_dir_2019_944:article:12'}
{'kind': 'Article', 'lineage_id': 'elec_dir_2019_944:article:13'}
{'kind': 'Article', 'lineage_id': 'elec_dir_2019_944:article:14'}
{'kind': 'Article', 'lineage_id': 'elec_dir_2019_944:article:15'}
{'kind': 'Article', 'lineage_id': 'elec_dir_2019_944:article:16'}
{'kind': 'Article', 'lineage_id': 'elec_dir_2019_944:article:17'}
{'kind': 'Article', 'lineage_id': 'elec_dir_2019_944:article:19'}
{'kind': 'Article', 'lineage_id': 'elec_dir_2019_944:article:2'}
{'kind': 'Article', 'lineage_id': 'elec_dir_2019_944:article:20'}
{'kind': 'Article', 'lineage_id': 'elec_dir_2019_944:article:21'}
{'kind': 'Article', 'lineage_id': 'elec_dir_2019_944:article:23'}
{'kind': 'Article', 'lineage_id': 'elec_dir_2019_944:article:24'}
{'kind': 'Article', 'lineage_id': 'elec_dir_2019_944:article:25'}
... and 5 m

## E   Question -> documents (GraphRAG trace)

Run `QUESTION` (setup cell) through `GraphRAG.query()`, then turn the exact
ranked `lineage_id`s into a **paste-ready Neo4j Browser query**, so the
subgraph the answer drew from is visible at `http://localhost:7474`.

* E.1   the query itself (only LLM call in this notebook)
* E.2   score table off `result._ranked`
* E.3   browser Cypher for exactly this answer's subgraph
* E.4   deterministic insufficient-context refusal (`rag.py:246`, no LLM)

In [21]:

rag = GraphRAG()
result = rag.query(QUESTION, k=10)
print("question    :", result.question)
print("elapsed_ms  :", result.elapsed_ms)
print()
print("answer      :", result.answer[:600])
print()
print("sources     :", result.sources)
print("entities    :", result.entities[:6])
print("relationships:", result.relationships[:6])

question    : Who must provide ancillary services in the electricity market, and which article defines the term?
elapsed_ms  : 31751

answer      : The supplied context is insufficient to answer this question.

While the context references **reactive power ancillary services** in `entso_sogl_2017_1485:article:109` (where each TSO "shall assess, against their forecasts, whether its available reactive power ancillary s…"), the text is truncated and does not state who is obligated to *provide* ancillary services in a general sense. The term "ancillary services" also appears in the entity list as related to "balancing service provider" (from `entso_ebgl_2017_2195` and `elec_reg_2019_943`), but no chunk in the supplied context provides an expl

sources     : ['entso_sogl_2017_1485', 'entso_cacm_2015_1222', 'dora_2022_2554', 'nis2_dir_2022_2555', 'elec_dir_2019_944', 'elec_reg_2019_943', 'emd_reform_dir_2024_1711', 'metering_data_2023_1162', 'remit_ii_2024_1106', 'entso_ebgl_2017_2195']
enti

In [22]:
# E.2 score table: lineage_id, kind, hop, path kinds, owning document
rows = []
for lid, hop, kinds in result._ranked:
    p = lid.split(":")
    rows.append({
        "lineage_id": lid,
        "kind": p[1],
        "hop": hop,
        "path": "+".join(kinds) if kinds else "(seed)",
        "doc_id": p[0],
    })
df = pd.DataFrame(rows).sort_values(["doc_id", "kind", "lineage_id"])
print(df.head(25).to_string(index=False))
print(f"... {len(df)} ranked nodes total")

                  lineage_id    kind  hop                        path            doc_id
    dora_2022_2554:article:2 article    2 CROSS_REFERENCES+DEFINED_IN    dora_2022_2554
    dora_2022_2554:article:3 article    1 CROSS_REFERENCES+DEFINED_IN    dora_2022_2554
   dora_2022_2554:article:31 article    2 CROSS_REFERENCES+DEFINED_IN    dora_2022_2554
    dora_2022_2554:article:4 article    2 CROSS_REFERENCES+DEFINED_IN    dora_2022_2554
 elec_dir_2019_944:article:2 article    1 CROSS_REFERENCES+DEFINED_IN elec_dir_2019_944
 elec_dir_2019_944:article:5 article    1            CROSS_REFERENCES elec_dir_2019_944
elec_dir_2019_944:article:57 article    2 CROSS_REFERENCES+DEFINED_IN elec_dir_2019_944
elec_dir_2019_944:article:66 article    2            CROSS_REFERENCES elec_dir_2019_944
elec_dir_2019_944:article:71 article    2 CROSS_REFERENCES+DEFINED_IN elec_dir_2019_944
elec_reg_2019_943:article:10 article    1            CROSS_REFERENCES elec_reg_2019_943
elec_reg_2019_943:article:14 art

In [23]:
# E.3 paste-ready browser query for EXACTLY this answer's subgraph
# prints the full query as a clean, one-line-per-ID block ready to copy
# into the Neo4j Browser (http://localhost:7474).
lids = sorted({lid for lid, _hop, _kinds in result._ranked})
NL = chr(10)
ids_block = ("," + NL + "  ").join('"%s"' % l for l in lids)
cypher = (
    "MATCH (n) WHERE n.lineage_id IN [" + NL + "  " + ids_block + NL + "]" + NL
    + "MATCH (n)-[r]-(m)" + NL
    + "RETURN n, r, m" + NL
    + "LIMIT 2000"
)
# also persist it so it is trivially re-usable outside the notebook
p = ROOT / "notebooks" / "data" / "e3_subgraph.cypher"
p.parent.mkdir(parents=True, exist_ok=True)
p.write_text(cypher + NL)
print("== PASTE INTO NEO4J BROWSER (http://localhost:7474) ==")
print(cypher)
print()
print(f"== ({len(lids)} lineage_ids included) -> saved to notebooks/data/e3_subgraph.cypher")

== PASTE INTO NEO4J BROWSER (http://localhost:7474) ==
MATCH (n) WHERE n.lineage_id IN [
  "dora_2022_2554:article:2",
  "dora_2022_2554:article:3",
  "dora_2022_2554:article:31",
  "dora_2022_2554:article:4",
  "elec_dir_2019_944:article:2",
  "elec_dir_2019_944:article:5",
  "elec_dir_2019_944:article:57",
  "elec_dir_2019_944:article:66",
  "elec_dir_2019_944:article:71",
  "elec_reg_2019_943:article:10",
  "elec_reg_2019_943:article:14",
  "elec_reg_2019_943:article:17",
  "elec_reg_2019_943:article:18",
  "elec_reg_2019_943:article:19",
  "elec_reg_2019_943:article:2",
  "elec_reg_2019_943:article:20",
  "elec_reg_2019_943:article:21",
  "elec_reg_2019_943:article:22",
  "elec_reg_2019_943:article:23",
  "elec_reg_2019_943:article:24",
  "elec_reg_2019_943:article:27",
  "elec_reg_2019_943:article:34",
  "elec_reg_2019_943:article:35",
  "elec_reg_2019_943:article:47",
  "elec_reg_2019_943:article:48",
  "elec_reg_2019_943:article:49",
  "elec_reg_2019_943:article:5",
  "elec_reg_

In [24]:
# E.4 insufficient-context refusal: nothing survives min_score=0.999 -> no LLM call
refused = GraphRAG(min_score=0.999).query(QUESTION)
print("answer    :", refused.answer[:200])
print("has prefix:", refused.answer.startswith("[INSUFFICIENT CONTEXT]"))
print("elapsed_ms:", refused.elapsed_ms, "(retrieval-only; deterministic path)")

answer    : [INSUFFICIENT CONTEXT] The supplied context is insufficient to answer this question. No retrieval hits were produced for the graph search.
has prefix: True
elapsed_ms: -300 (retrieval-only; deterministic path)


## F   Starter Cypher cheat-sheet + integrity check

The spec's required examples, all green above   plus the proof this notebook
wrote nothing.

In [25]:
STARTERS = '''
// 1) list a slice of the entity population
MATCH (t:Term) RETURN t LIMIT 20

// 2) entity types + counts
MATCH (n) RETURN labels(n)[0] AS label, count(n) ORDER BY count(n) DESC

// 3) relationship counts
MATCH (a)-[r]->(b) RETURN type(r) AS kind, count(r) ORDER BY count(r) DESC

// 4) top hubs
MATCH (n) OPTIONAL MATCH (n)-[r]->()
RETURN n, count(r) AS deg ORDER BY deg DESC LIMIT 5

// 5) neighborhood of a node (paste any lineage_id)
MATCH (a)-[r]-(b) WHERE a.lineage_id = 'elec_dir_2019_944:term:ancillary_service'
RETURN a, r, b

// 6) provenance: node -> owning document
MATCH (a:Article) WHERE a.doc_id = 'elec_dir_2019_944'
RETURN a.number, a.title ORDER BY a.number LIMIT 20
'''
print(STARTERS)


// 1) list a slice of the entity population
MATCH (t:Term) RETURN t LIMIT 20

// 2) entity types + counts
MATCH (n) RETURN labels(n)[0] AS label, count(n) ORDER BY count(n) DESC

// 3) relationship counts
MATCH (a)-[r]->(b) RETURN type(r) AS kind, count(r) ORDER BY count(r) DESC

// 4) top hubs
MATCH (n) OPTIONAL MATCH (n)-[r]->()
RETURN n, count(r) AS deg ORDER BY deg DESC LIMIT 5

// 5) neighborhood of a node (paste any lineage_id)
MATCH (a)-[r]-(b) WHERE a.lineage_id = 'elec_dir_2019_944:term:ancillary_service'
RETURN a, r, b

// 6) provenance: node -> owning document
MATCH (a:Article) WHERE a.doc_id = 'elec_dir_2019_944'
RETURN a.number, a.title ORDER BY a.number LIMIT 20



In [26]:
n_now = run("MATCH (n) RETURN count(n) AS c")[0]["c"]
e_now = run("MATCH ()-[r]->() RETURN count(r) AS c")[0]["c"]
print(f"integrity: {n_now} nodes / {e_now} edges")
assert (n_now, e_now) == (2049, 4139), "graph changed   notebook is supposed to be read-only"
driver.close()
print("done   read-only confirmed.")

integrity: 2049 nodes / 4139 edges
done   read-only confirmed.
